In [9]:
from datasets import load_dataset

dataset = load_dataset("Yelp/yelp_review_full")

In [10]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [11]:
dataset['train'][10086]

{'label': 3,
 'text': 'So I used the \\"CHAT ONLINE with an agent\\" option today.  You can find this link near the top, right side of the screen on the main Overview page on thier website.  I\'ve used this feature with other organizations in the past - so it wasn\'t really new for me.  I chatted with Vivian and she answered my quick question about dog fees in a timely manner.  That\'s all I could ask for.  Thanks Vivian!  \\n\\nSince I wasn\'t able to find \\"Dog Fee\\" information on the website, I offer the info here:\\n\\n\\"Pets allowed with nonrefundable fee of 75 USD per stay. No larger than 80 lbs. Pet agreement must be signed at check in. Call hotel for details. Record of complete and up to date vaccinations required.\\"\\n\\nI hope this helps all the dog lovers out there.  :)'}

In [4]:
import random
import pandas as pd
import datasets
from IPython.display import display, HTML

In [5]:
def show_random_elements(dataset, num_examples=1):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [13]:
show_random_elements(tokenized_datasets["train"], num_examples=1)

,label,text,input_ids,token_type_ids,attention_mask
0,3 stars,"We stayed here as a waystation, not a destination, so we didn't spend time enjoying the facilities or the casino. Front desk personnel were not very pleasant or helpful at all, but every other person on staff we encountered was wonderful - the valets in the evening and in the morning, the room service woman who took our order and the guy who delivered it, and the housekeeper who came for the turn down service. The room was spacious and comfortable, but a little dated. The room service food (pizza and milk and cookies) was tasty, generously-sized, and reasonably priced. There was a prominent legend on the bill that said \""service charge included\"". We know Vegas and therefore knew this hotel was not near the Strip, so the distance to it did not come as a surprise. I would stay here again notwithstanding the surly 20-something at the front desk. My husband was serioulsly annoyed that you had to pay $10 for 24-hours of WiFi since WiFi had been free everywhere else on our rather lengthy road trip, even at the Motel-6-ish places.","[101, 1284, 3523, 1303, 1112, 170, 3242, 7984, 117, 1136, 170, 7680, 117, 1177, 1195, 1238, 112, 189, 4511, 1159, 8965, 1103, 3380, 1137, 1103, 14330, 119, 5967, 3917, 4675, 1127, 1136, 1304, 10287, 1137, 14739, 1120, 1155, 117, 1133, 1451, 1168, 1825, 1113, 2546, 1195, 8181, 1108, 7310, 118, 1103, 191, 26391, 1116, 1107, 1103, 3440, 1105, 1107, 1103, 2106, 117, 1103, 1395, 1555, 1590, 1150, 1261, 1412, 1546, 1105, 1103, 2564, 1150, 4653, 1122, 117, 1105, 1103, 26458, 1150, 1338, 1111, 1103, 1885, 1205, 1555, 119, 1109, 1395, 1108, 25274, 1105, 6062, 117, 1133, 170, 1376, 5422, 119, ...]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...]"


In [14]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(10000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

In [15]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions= predictions, references=labels)

In [18]:
from transformers import TrainingArguments

model_dir = "models/bert-base-cased-finetune-yelp"

# logging_steps 默认值为500，根据我们的训练数据和步长，将其设置为100
training_args = TrainingArguments(
      output_dir=model_dir,
      per_device_train_batch_size=16,
      num_train_epochs=3,
      logging_steps=30
)

In [19]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

In [20]:
trainer.train()

Step,Training Loss
30,1.528199
60,1.259524
90,1.177165
120,1.164383
150,1.118768
180,1.065471
210,1.054542
240,1.116115
270,1.006713
300,1.031538


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=0.7499831548055013, metrics={'train_runtime': 955.0569, 'train_samples_per_second': 31.412, 'train_steps_per_second': 1.963, 'total_flos': 7893544273920000.0, 'train_loss': 0.7499831548055013, 'epoch': 3.0})

In [21]:
small_test_dataset = tokenized_datasets["test"].shuffle(seed=64).select(range(500))

In [22]:
trainer.evaluate(small_test_dataset)

Training Loss,Validation Loss,Step,Accuracy
0.463505,1.077342,1875,0.624000


{'eval_loss': 1.0773420333862305, 'eval_accuracy': 0.624}

In [ ]:
trainer.save_model(model_dir)